# AO3 Tag Analysis

Notebook version of `ao3_tag_analysis.py`. Reads an existing `ao3_tag_metadata.csv`
(produced by `ao3_tag_scraper.py`) and runs two further analyses beyond
`ao3_tag_visualizer.py`'s bipartite network and tag-pair lift/PMI network:

1. **additional_tags frequency ranking** -- which values are most common, and
   which are least common (excluding one-off singletons)
2. **Cross-field hierarchical clustering** -- pools labels from *all* metadata
   fields (`rating`, `warnings`, `category`, `fandom`, `relationship`,
   `character`, `additional_tags`) and clusters them by lift/PMI similarity,
   rendered as a dendrogram + reordered heatmap plus a discrete
   cluster-membership CSV.

No network access is required -- this only reads a local CSV. Functions
reused from `ao3_tag_visualizer.py`'s tag-pair-statistics machinery are
copied inline below (matching this repo's existing notebook convention),
each marked with a comment noting the source -- keep them in sync if that
file changes.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present
# (pandas, numpy, scipy, seaborn, matplotlib). Safe to re-run.
%pip install -q pandas numpy scipy seaborn matplotlib

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.sparse as sp
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

## Configuration

Edit these values, then run all cells in order.

In [ ]:
INPUT = "ao3_tag_metadata.csv"

FREQUENCY_TOP_N = 20          # most frequent additional_tags to report, per
                              # category -- seed tags and non-seed tags each
                              # get up to this many
FREQUENCY_BOTTOM_N = 20       # least frequent additional_tags to report
FREQUENCY_MIN_COUNT = 2       # floor for "least frequent" -- excludes values with
                              # fewer works than this (default 2 excludes one-off
                              # singletons)
FREQUENCY_OUT = "ao3_additional_tags_frequency.csv"

TOP_TAGS = 60                 # top N tags overall, pooled across all 7 metadata
                              # fields, by document frequency, before clustering.
                              # Overridden by ALL_TAGS
ALL_TAGS = False              # cluster using every tag from all 7 metadata fields,
                              # ignoring TOP_TAGS
MIN_PAIR_COUNT = 2            # drop pairs co-occurring fewer than this many times
                              # before clustering
N_CLUSTERS = 10               # cut the dendrogram into up to this many discrete
                              # clusters -- a ceiling rather than a fixed target when
                              # MIN_CLUSTER_SIZE is set
MIN_CLUSTER_SIZE = 1          # merge/reduce clusters smaller than this; NUM_CLUSTERS
                              # becomes a ceiling rather than a fixed target when this
                              # is > 1 (1 = no minimum)
CLUSTER_METHOD = "average"    # scipy linkage method: average/complete/ward/single
HEATMAP_OUT_DIR = "heatmaps"
CLUSTER_HEATMAP_OUT = None    # None -> f"{HEATMAP_OUT_DIR}/heatmap_clusters.png"
CLUSTERS_OUT = "ao3_tag_clusters.csv"

DELIMITER = ", "
ALL_METADATA_FIELDS = ["rating", "warnings", "category", "fandom",
                        "relationship", "character", "additional_tags"]

## Load metadata

The CSV is loaded once. Multi-value fields (e.g. `additional_tags`, `character`) are comma-delimited within a cell -- `explode_field` splits them into one row per (work, value) pair on demand.

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    return [v.strip() for v in cell.split(delimiter) if v.strip()]


def explode_field(df, field):
    exploded = df[["work_id", field]].copy()
    exploded[field] = exploded[field].apply(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


df = load_metadata(INPUT)
df.head()

## additional_tags frequency ranking

A flat frequency count of every `additional_tags` value across the whole dataset -- the most common values, and the least common ones (excluding one-off singletons per `FREQUENCY_MIN_COUNT`).

In [ ]:
# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def additional_tags_frequency(df, min_bottom_count=2):
    # Seed tags are the values in df["tag"] (the AO3 tag actually searched
    # to find each work). An additional_tags value that also happens to be
    # a seed tag partly reflects the scrape's own search bias rather than a
    # genuinely emergent discovery, so it's split out from non-seed values.
    seed_tags = set(df["tag"].unique())
    exploded = explode_field(df, "additional_tags")
    counts = exploded["additional_tags"].value_counts()
    counts = counts.reset_index()
    counts.columns = ["additional_tags", "count"]

    def sort_desc(c):
        return c.sort_values(["count", "additional_tags"], ascending=[False, True])

    is_seed = counts["additional_tags"].isin(seed_tags)
    most_frequent_seed = sort_desc(counts[is_seed])
    most_frequent_non_seed = sort_desc(counts[~is_seed])

    least_frequent = counts[counts["count"] >= min_bottom_count]
    least_frequent = least_frequent.sort_values(["count", "additional_tags"],
                                                 ascending=[True, True])
    return most_frequent_seed, most_frequent_non_seed, least_frequent


def write_frequency_csv(most_frequent_seed, most_frequent_non_seed, least_frequent,
                         top_n, bottom_n, out_path):
    most_seed = most_frequent_seed.head(top_n).copy()
    most_seed["rank_type"] = "most_frequent_seed_tag"
    most_non_seed = most_frequent_non_seed.head(top_n).copy()
    most_non_seed["rank_type"] = "most_frequent_non_seed_tag"
    least = least_frequent.head(bottom_n).copy()
    least["rank_type"] = "least_frequent"
    combined = pd.concat([most_seed, most_non_seed, least], ignore_index=True)
    combined = combined[["rank_type", "additional_tags", "count"]]
    combined.to_csv(out_path, index=False)
    return combined


most_frequent_seed, most_frequent_non_seed, least_frequent = additional_tags_frequency(
    df, min_bottom_count=FREQUENCY_MIN_COUNT)
frequency_table = write_frequency_csv(most_frequent_seed, most_frequent_non_seed, least_frequent,
                                       FREQUENCY_TOP_N, FREQUENCY_BOTTOM_N, FREQUENCY_OUT)
print(f"wrote {FREQUENCY_OUT} "
      f"({min(FREQUENCY_TOP_N, len(most_frequent_seed))} most frequent seed tags, "
      f"{min(FREQUENCY_TOP_N, len(most_frequent_non_seed))} most frequent non-seed tags, "
      f"{min(FREQUENCY_BOTTOM_N, len(least_frequent))} least frequent)")
frequency_table

## Cross-field hierarchical clustering (lift/PMI)

A different question from frequency ranking: not "how common is this value", but
"which labels -- of *any* metadata field, `rating`/`warnings`/`category` included --
statistically tend to appear together". Pools all seven fields into one namespaced
label space (`f"{field}::{value}"`, same scheme as `ao3_tag_visualizer.py`'s
`--tag-pairs`), computes pairwise lift/PMI:

- `lift(A, B) = P(A, B) / (P(A) * P(B))`
- `pmi(A, B) = log2(lift(A, B))`

...then hierarchically clusters the resulting tag x tag PMI matrix (via
`scipy`/`seaborn.clustermap`) to group labels by similarity, rather than just
ranking individual pairs. `apply_min_pair_count` still runs first (dropping
low-sample coincidences whose lift is statistically meaningless), but unlike
`--tag-pairs`' `--min-pmi`/`--max-pmi` "boring middle" exclusion, clustering
keeps the *full* similarity structure, including near-zero PMI values --
that's what a dendrogram needs.

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def build_document_tag_table(df, fields=ALL_METADATA_FIELDS):
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    tables = []
    for field in fields:
        exploded = explode_field(deduped, field)
        exploded = exploded.rename(columns={field: "value"})
        exploded["tag_id"] = field + "::" + exploded["value"]
        tables.append(exploded[["work_id", "tag_id"]])
    return pd.concat(tables, ignore_index=True)


def top_k_tags_by_document_frequency(tag_table, k):
    if k is None:
        return None
    counts = tag_table["tag_id"].value_counts().reset_index()
    counts.columns = ["tag_id", "count"]
    counts = counts.sort_values(["count", "tag_id"], ascending=[False, True])
    return set(counts["tag_id"].head(k))


def build_tag_incidence_matrix(tag_table, keep_tags):
    filtered = tag_table[tag_table["tag_id"].isin(keep_tags)]
    columns = sorted(keep_tags)
    work_ids = sorted(filtered["work_id"].unique())
    col_index = {tag_id: i for i, tag_id in enumerate(columns)}
    row_index = {work_id: i for i, work_id in enumerate(work_ids)}

    row_idx = filtered["work_id"].map(row_index).to_numpy()
    col_idx = filtered["tag_id"].map(col_index).to_numpy()
    matrix = sp.coo_matrix(
        (np.ones(len(filtered), dtype=np.int8), (row_idx, col_idx)),
        shape=(len(work_ids), len(columns)), dtype=np.int8,
    ).tocsr()
    # Collapse any duplicate (work_id, tag_id) rows to a plain 0/1 presence
    # flag -- matches pd.crosstab(...) > 0's semantics regardless of raw
    # counts (coo_matrix sums duplicate entries on conversion to csr).
    matrix.data[:] = 1
    return pd.DataFrame.sparse.from_spmatrix(matrix, index=work_ids, columns=columns)


def tag_pair_statistics(incidence, n_docs):
    tags = list(incidence.columns)
    values = incidence.sparse.to_coo().tocsr().astype(np.int64)
    joint = values.T @ values
    marginal = np.asarray(joint.diagonal()).ravel()

    # scipy.sparse.triu only stores the upper triangle's nonzero entries --
    # equivalent to np.triu_indices(k=1) on the dense matrix, but without
    # ever materializing the full tags x tags array to get there (a dense
    # tags x tags int64 array is 48.9 GiB at 81,037 tags -- exactly the
    # MemoryError this sparse rewrite fixes).
    upper = sp.triu(joint, k=1).tocoo()
    i_idx, j_idx, joint_counts = upper.row, upper.col, upper.data
    nonzero = joint_counts > 0
    i_idx, j_idx, joint_counts = i_idx[nonzero], j_idx[nonzero], joint_counts[nonzero]

    count_a = marginal[i_idx]
    count_b = marginal[j_idx]
    lift = (joint_counts * n_docs) / (count_a * count_b)
    pmi = np.log2(lift)

    # Explicit canonicalization: tag_a is always the alphabetically-smaller
    # of the pair, regardless of the incidence matrix's own column order --
    # np.triu_indices above only guarantees i < j by column *position*, not
    # by alphabetical value. This used to be true only as an incidental side
    # effect of build_tag_incidence_matrix always pre-sorting its columns;
    # enforcing it here means the guarantee holds for any caller, not just
    # today's one. lift/pmi are symmetric in count_a/count_b already, so
    # only the names and their paired counts need to swap together.
    tags_arr = np.array(tags)
    a_names = tags_arr[i_idx]
    b_names = tags_arr[j_idx]
    swap = a_names > b_names
    tag_a = np.where(swap, b_names, a_names)
    tag_b = np.where(swap, a_names, b_names)
    count_a_final = np.where(swap, count_b, count_a)
    count_b_final = np.where(swap, count_a, count_b)

    return pd.DataFrame({
        "tag_a": tag_a,
        "tag_b": tag_b,
        "joint_count": joint_counts,
        "count_a": count_a_final,
        "count_b": count_b_final,
        "lift": lift,
        "pmi": pmi,
    })


def apply_min_pair_count(pair_stats, min_pair_count):
    return pair_stats[pair_stats["joint_count"] >= min_pair_count]


def tag_pair_matrix(pair_stats, tags, value_col="pmi"):
    ordered = sorted(tags)
    matrix = pd.DataFrame(np.nan, index=ordered, columns=ordered)
    for _, row in pair_stats.iterrows():
        if row["tag_a"] in matrix.index and row["tag_b"] in matrix.index:
            matrix.loc[row["tag_a"], row["tag_b"]] = row[value_col]
            matrix.loc[row["tag_b"], row["tag_a"]] = row[value_col]
    return matrix


# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def build_all_fields_pair_data(df, top_tags, min_pair_count):
    tag_table = build_document_tag_table(df, fields=ALL_METADATA_FIELDS)
    keep_tags = top_k_tags_by_document_frequency(tag_table, top_tags)
    if keep_tags is None:
        keep_tags = set(tag_table["tag_id"].unique())
    incidence = build_tag_incidence_matrix(tag_table, keep_tags)
    n_docs = df["work_id"].nunique()
    pair_stats = tag_pair_statistics(incidence, n_docs)
    pair_stats = apply_min_pair_count(pair_stats, min_pair_count)
    return pair_stats, keep_tags


top_tags_value = None if ALL_TAGS else TOP_TAGS
pair_stats, keep_tags = build_all_fields_pair_data(df, top_tags_value, MIN_PAIR_COUNT)
print(f"{len(keep_tags)} tags kept, {len(pair_stats)} surviving pairs")

## Clustermap + cluster membership CSV

`fill_matrix` replaces `display_matrix`'s NaN cells (missing/filtered pairs) with `0.0` for clustering input only -- scipy's linkage computation can't handle NaN, and treating "no observed pair" as "independent" is a deliberate simplification scoped to this step. The rendered heatmap still masks true-NaN cells blank via `display_matrix`, so the fill never shows up as a fabricated data point. The same linkage matrix is used for both the plotted dendrogram and the cluster-membership cut, so the PNG's tree and the CSV's clusters can never disagree with each other.

In [ ]:
# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def build_cluster_matrix(pair_stats, keep_tags):
    display_matrix = tag_pair_matrix(pair_stats, keep_tags, value_col="pmi")
    fill_matrix = display_matrix.fillna(0.0)
    return display_matrix, fill_matrix


def compute_linkage(fill_matrix, method="average"):
    if fill_matrix.shape[0] < 2:
        return None
    distances = pdist(fill_matrix.to_numpy(), metric="euclidean")
    return linkage(distances, method=method)


def cut_clusters(linkage_matrix, tags, n_clusters, min_cluster_size=1):
    cluster_ids = None
    for k in range(min(n_clusters, len(tags)), 0, -1):
        candidate = fcluster(linkage_matrix, t=k, criterion="maxclust")
        if k == 1 or pd.Series(candidate).value_counts().min() >= min_cluster_size:
            cluster_ids = candidate
            break
    rows = []
    for tag_id, cluster_id in zip(tags, cluster_ids):
        field, _, label = tag_id.partition("::")
        rows.append({"tag_id": tag_id, "field": field, "label": label,
                      "cluster_id": int(cluster_id)})
    result = pd.DataFrame(rows, columns=["tag_id", "field", "label", "cluster_id"])
    return result.sort_values(["cluster_id", "tag_id"]).reset_index(drop=True)


def render_cluster_heatmap(display_matrix, fill_matrix, linkage_matrix, out_path):
    if fill_matrix.empty or linkage_matrix is None:
        print("skipping cluster heatmap: fewer than 2 tags after filtering")
        return None
    width = max(10, 0.35 * fill_matrix.shape[1] + 4)
    height = max(8, 0.3 * fill_matrix.shape[0] + 4)
    annot = fill_matrix.shape[0] * fill_matrix.shape[1] <= 500
    grid = sns.clustermap(
        fill_matrix, row_linkage=linkage_matrix, col_linkage=linkage_matrix,
        mask=display_matrix.isna(), cmap="coolwarm", center=0,
        annot=annot, fmt=".2f", figsize=(width, height),
        cbar_kws={"label": "PMI (log2 lift)"},
    )
    grid.savefig(out_path, dpi=150)
    print(f"wrote {out_path} ({fill_matrix.shape[0]}x{fill_matrix.shape[1]})")
    return grid


display_matrix, fill_matrix = build_cluster_matrix(pair_stats, keep_tags)
linkage_matrix = compute_linkage(fill_matrix, method=CLUSTER_METHOD)

os.makedirs(HEATMAP_OUT_DIR, exist_ok=True)
cluster_heatmap_out = CLUSTER_HEATMAP_OUT or os.path.join(HEATMAP_OUT_DIR, "heatmap_clusters.png")
grid = render_cluster_heatmap(display_matrix, fill_matrix, linkage_matrix, cluster_heatmap_out)

if linkage_matrix is not None:
    clusters_df = cut_clusters(linkage_matrix, list(fill_matrix.index), N_CLUSTERS,
                            MIN_CLUSTER_SIZE)
    clusters_df.to_csv(CLUSTERS_OUT, index=False)
    print(f"wrote {CLUSTERS_OUT} ({len(clusters_df)} tags, "
          f"{clusters_df['cluster_id'].nunique()} clusters)")
clusters_df.head(20)

## Done

`{FREQUENCY_OUT}`, `{CLUSTER_HEATMAP_OUT or f"{HEATMAP_OUT_DIR}/heatmap_clusters.png"}`,
and `{CLUSTERS_OUT}` are now in the notebook's working directory. Re-run from the
Configuration cell with different `FREQUENCY_TOP_N`/`FREQUENCY_BOTTOM_N`/
`FREQUENCY_MIN_COUNT`/`TOP_TAGS`/`MIN_PAIR_COUNT`/`N_CLUSTERS`/`CLUSTER_METHOD`
values to adjust what's included.